# k=64 with MSE + NMSE + Correlation Loss

Adds the spectral-correlation term on top of the standard MSE+NMSE mixed loss so all three signals are jointly minimised.

This notebook is a standalone, simplified version of `adjscc_k64_correlation_mse_loss.py` and follows the same flow as `4 feb/ADJSCC-CSInet+.ipynb`:
1. dataset
2. AF module
3. ATN module
4. encoder
5. real → complex symbols + power normalisation
6. wireless channel
7. complex → real (C2R)
8. decoder
9. STN
10. training loop


## Imports and seed

In [ ]:
import math
import os
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Config

In [ ]:
class TrainingConfig:
    train_file: str = "train_data.mat"
    val_file: str = "val_data.mat"
    test_file: str = "test_data.mat"
    checkpoint_dir: str = "checkpoints_k64_correlation_mse_loss"
    fast_dev_run: bool = False
    run_training: bool = True
    epochs: int = 500
    batch_size: int = 200
    learning_rate: float = 0.001
    min_lr: float = 0.0001
    patience: int = 20
    weight_decay: float = 1e-05
    grad_clip: float = 1.0
    snr_low: float = -10.0
    snr_high: float = 10.0
    k_feedback: int = 64
    compression_ratio: int = 16
    mse_weight: float = 0.4
    nmse_weight: float = 0.4
    corr_weight: float = 0.2
    warmup_epochs: int = 20
    save_every: int = 10
    run_evaluation: bool = True
    checkpoint_path: str | None = None
    resume_latest: bool = False
cfg = TrainingConfig()
os.makedirs(cfg.checkpoint_dir, exist_ok=True)


## Dataset

Streams the QuaDRiGa CSI HDF5 files lazily and applies a global per-channel scale computed on the train split.

In [ ]:
def _read_scalar(dataset):
    value = dataset[()]
    if isinstance(value, np.ndarray) and value.size == 1:
        return float(value.reshape(-1)[0])
    return value

def load_dataset_cfg(path: str):
    with h5py.File(path, "r") as f:
        group = f["cfg"]
        out = {key: _read_scalar(group[key]) for key in group.keys() if isinstance(group[key], h5py.Dataset)}
    return out

def get_train_global_scale(train_path, chunk_size=500):
    stats = {"dl": {"sum_sq": 0.0, "count": 0}, "ul": {"sum_sq": 0.0, "count": 0}}
    with h5py.File(train_path, "r") as f:
        for key, short_name in [("csi_dl", "dl"), ("csi_ul", "ul")]:
            dataset = f[key]
            num_samples = dataset.shape[3]
            for start in range(0, num_samples, chunk_size):
                stop = min(start + chunk_size, num_samples)
                chunk = dataset[:, :, :, start:stop]
                real = chunk["real"].astype(np.float32)
                imag = chunk["imag"].astype(np.float32)
                stats[short_name]["sum_sq"] += float(np.sum(real ** 2) + np.sum(imag ** 2))
                stats[short_name]["count"] += real.size + imag.size

    out = {}
    for key in ("dl", "ul"):
        var = stats[key]["sum_sq"] / max(stats[key]["count"], 1)
        out[key] = {"std": float(np.sqrt(var + 1e-12))}
    print("Train-only scale stats:", out)
    return out

class CSIDatasetManager:
    def __init__(self, train_path, val_path, test_path, stats):
        self.stats = stats
        self.paths = {"train": train_path, "val": val_path, "test": test_path}
        self.files = {}
        self.datasets = {}
        self.lengths = {}

        for split, path in self.paths.items():
            handle = h5py.File(path, "r")
            self.files[split] = handle
            self.datasets[split] = {"dl": handle["csi_dl"], "ul": handle["csi_ul"]}
            self.lengths[split] = int(handle["csi_dl"].shape[3])
            print(f"{split} samples: {self.lengths[split]}")

    def close(self):
        for handle in self.files.values():
            handle.close()

    def _normalize(self, arr, key):
        return arr / (self.stats[key]["std"] + 1e-8)

    def denormalize(self, tensor, key):
        return tensor * (self.stats[key]["std"] + 1e-8)

    def _process(self, batch_arr, key, normalize=True):
        real = batch_arr["real"].astype(np.float32)
        imag = batch_arr["imag"].astype(np.float32)
        if normalize:
            real = self._normalize(real, key)
            imag = self._normalize(imag, key)
        merged = np.stack([real, imag], axis=2)
        merged = np.squeeze(merged, axis=3)
        merged = np.transpose(merged, (3, 2, 0, 1))
        return merged

    def get_batch(self, split, indices, snr_values=None):
        indices = np.asarray(indices, dtype=np.int64)
        indices.sort()
        dl_raw = self.datasets[split]["dl"][:, :, :, indices]
        ul_raw = self.datasets[split]["ul"][:, :, :, indices]

        dl = torch.from_numpy(self._process(dl_raw, "dl", normalize=True)).float()
        ul = torch.from_numpy(self._process(ul_raw, "ul", normalize=True)).float()

        if snr_values is None:
            snr_values = np.random.uniform(cfg.snr_low, cfg.snr_high, size=(len(indices), 1)).astype(np.float32)
        else:
            snr_values = np.asarray(snr_values, dtype=np.float32).reshape(len(indices), 1)

        snr = torch.from_numpy(snr_values).float()
        return dl, ul, snr

    def iterate_split(self, split, batch_size, shuffle=False, generator=None, fixed_snr=None):
        total = self.lengths[split]
        order = np.arange(total, dtype=np.int64)
        if shuffle:
            rng = generator if generator is not None else np.random.default_rng()
            rng.shuffle(order)

        for start in range(0, total, batch_size):
            batch_indices = order[start:start + batch_size]
            snr_values = None
            if fixed_snr is not None:
                snr_values = np.full((len(batch_indices), 1), fixed_snr, dtype=np.float32)
            yield self.get_batch(split, batch_indices, snr_values=snr_values)

In [ ]:
# Set these paths to the QuaDRiGa CSI .mat files on your machine.
train_file = "train_data.mat"
val_file   = "val_data.mat"
test_file  = "test_data.mat"


In [ ]:
stats = get_train_global_scale(train_file)
dataset = CSIDatasetManager(train_file, val_file, test_file, stats)


## AF Module

Channel-wise SNR-aware feature recalibration: GAP over (H,W), concat with SNR (dB), 2-layer MLP → sigmoid → per-channel scale.

In [ ]:
class AFModule(nn.Module):
    def __init__(self, channels, reduction_ratio=2):
        super().__init__()
        hidden_dim = max(channels // reduction_ratio, 1)
        self.fc1 = nn.Linear(channels + 1, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, channels)

    def forward(self, x, snr):
        pooled = F.adaptive_avg_pool2d(x, 1).flatten(1)
        scale = torch.cat([pooled, snr], dim=1)
        scale = F.relu(self.fc1(scale))
        scale = torch.sigmoid(self.fc2(scale)).view(x.size(0), x.size(1), 1, 1)
        return x * scale

## ATN — Analysis Transform Network

Three-layer (or wider, in deeper variants) conv stack with asymmetric strides that compresses the 32×32 angular-delay map to the truncated representation used by the SC-CSI encoder.

In [ ]:
class ATN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 16, kernel_size=3, stride=(2, 1), padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.prelu1 = nn.PReLU()
        self.af1 = AFModule(16)

        self.conv2 = nn.Conv2d(16, 16, kernel_size=3, stride=(2, 1), padding=1)
        self.bn2 = nn.BatchNorm2d(16)
        self.prelu2 = nn.PReLU()
        self.af2 = AFModule(16)

        self.conv3 = nn.Conv2d(16, 2, kernel_size=3, stride=(2, 1), padding=1)
        self.bn3 = nn.BatchNorm2d(2)

    def forward(self, x, snr):
        x = self.af1(self.prelu1(self.bn1(self.conv1(x))), snr)
        x = self.af2(self.prelu2(self.bn2(self.conv2(x))), snr)
        x = self.bn3(self.conv3(x))
        return x

## Encoder — CSINet+ encoder with AF modules

Two 7×7 conv blocks with AF modules, then a fully-connected layer projects the flattened map to the M-dimensional real-valued codeword.

In [ ]:
class CsiNetPlusEncoderWithAF(nn.Module):
    def __init__(self, compression_ratio):
        super().__init__()
        self.input_channels = 2
        self.height = 32
        self.width = 32
        self.total_elements = self.input_channels * self.height * self.width
        self.M = self.total_elements // compression_ratio

        self.conv1 = nn.Conv2d(2, 2, kernel_size=7, stride=1, padding=3)
        self.bn1 = nn.BatchNorm2d(2)
        self.af1 = AFModule(2)

        self.conv2 = nn.Conv2d(2, 2, kernel_size=7, stride=1, padding=3)
        self.bn2 = nn.BatchNorm2d(2)
        self.af2 = AFModule(2)

        self.fc = nn.Linear(self.total_elements, self.M)

    def forward(self, x, snr):
        x = self.af1(F.leaky_relu(self.bn1(self.conv1(x)), negative_slope=0.3), snr)
        x = self.af2(F.leaky_relu(self.bn2(self.conv2(x)), negative_slope=0.3), snr)
        x = x.flatten(1)
        return self.fc(x)

## Real → complex symbols + power normalisation

Splits the M real outputs into an M/2-length complex vector and rescales it to unit average power per symbol.

In [ ]:
def enc_to_complex_and_normalize(encoder_output):
    k = encoder_output.shape[1] // 2
    real_part = encoder_output[:, :k]
    imag_part = encoder_output[:, k:]
    s = torch.complex(real_part, imag_part)
    power = torch.mean(s.abs().square(), dim=1, keepdim=True)
    return s / torch.sqrt(power + 1e-8)

## Wireless channel

Differentiable OFDM AWGN channel: picks `k` uplink subcarriers, transmits the complex symbols, adds Gaussian noise scaled to the requested SNR, and applies maximum-ratio combining at the BS.

In [ ]:
class WirelessChannelSimulator(nn.Module):
    def __init__(self, num_bs_antennas=32, training_random_subcarriers=True):
        super().__init__()
        self.Nt = num_bs_antennas
        self.training_random_subcarriers = training_random_subcarriers

    def _select_indices(self, num_subcarriers, k, device):
        if self.training and self.training_random_subcarriers:
            return torch.randperm(num_subcarriers, device=device)[:k]
        return torch.linspace(0, num_subcarriers - 1, steps=k, device=device).round().long()

    def forward(self, s, snr_db, h_uplink_raw):
        batch_size, k = s.shape
        device = s.device

        num_subcarriers = h_uplink_raw.shape[2]
        if num_subcarriers < k:
            raise ValueError(f"Need at least {k} subcarriers, found {num_subcarriers}")

        indices = self._select_indices(num_subcarriers, k, device)
        h_sliced = h_uplink_raw[:, :, indices, :]
        h_u = torch.complex(h_sliced[:, 0], h_sliced[:, 1])

        snr_linear = torch.pow(10.0, snr_db / 10.0)
        noise_power = 1.0 / snr_linear
        noise_std = torch.sqrt(noise_power / 2.0).unsqueeze(-1)

        z_real = torch.randn(batch_size, k, self.Nt, device=device) * noise_std
        z_imag = torch.randn(batch_size, k, self.Nt, device=device) * noise_std
        z = torch.complex(z_real, z_imag)

        y = h_u * s.unsqueeze(-1) + z
        w = h_u / (torch.norm(h_u, dim=2, keepdim=True) + 1e-8)
        return torch.sum(torch.conj(w) * y, dim=2)

## C2R — Complex → real for the decoder

In [ ]:
class ComplexToReal(nn.Module):
    def forward(self, s_hat):
        return torch.cat([s_hat.real, s_hat.imag], dim=1)

## Decoder — CSINet+ RefineNet stack

FC → 32×32 feature map, an initial conv block, then a chain of RefineNet residual blocks (each conv block is followed by an AF module).

In [ ]:
class ModifiedRefineNetBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, 8, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm2d(8)
        self.af1 = AFModule(8)

        self.conv2 = nn.Conv2d(8, 16, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm2d(16)
        self.af2 = AFModule(16)

        self.conv3 = nn.Conv2d(16, channels, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(channels)
        self.af3 = AFModule(channels)

    def forward(self, x, snr):
        residual = x
        x = self.af1(F.leaky_relu(self.bn1(self.conv1(x)), negative_slope=0.3), snr)
        x = self.af2(F.leaky_relu(self.bn2(self.conv2(x)), negative_slope=0.3), snr)
        x = self.af3(self.bn3(self.conv3(x)), snr)
        return residual + x

In [ ]:
class CsiNetPlusDecoder(nn.Module):
    def __init__(self, input_dim, height=32, width=32, channels=2, num_blocks=5):
        super().__init__()
        self.height = height
        self.width = width
        self.channels = channels
        self.flattened_dim = height * width * channels

        self.fc = nn.Linear(input_dim, self.flattened_dim)
        self.initial_conv = nn.Conv2d(channels, channels, kernel_size=7, padding=3)
        self.initial_bn = nn.BatchNorm2d(channels)
        self.initial_af = AFModule(channels)
        self.refinenet_chain = nn.ModuleList([ModifiedRefineNetBlock(channels) for _ in range(num_blocks)])

    def forward(self, x, snr):
        x = self.fc(x).view(-1, self.channels, self.height, self.width)
        x = self.initial_af(F.leaky_relu(self.initial_bn(self.initial_conv(x)), negative_slope=0.3), snr)
        for block in self.refinenet_chain:
            x = block(x, snr)
        return x

## STN — Synthesis Transform Network

Mirror image of the ATN: transposed-conv stack that expands the latent back to the 32×32 angular-delay map.

In [ ]:
class STN(nn.Module):
    def __init__(self):
        super().__init__()
        self.trans_conv1 = nn.ConvTranspose2d(2, 16, kernel_size=3, stride=(2, 1), padding=1, output_padding=(1, 0))
        self.bn1 = nn.BatchNorm2d(16)
        self.prelu1 = nn.PReLU()
        self.af1 = AFModule(16)

        self.trans_conv2 = nn.ConvTranspose2d(16, 16, kernel_size=3, stride=(2, 1), padding=1, output_padding=(1, 0))
        self.bn2 = nn.BatchNorm2d(16)
        self.prelu2 = nn.PReLU()
        self.af2 = AFModule(16)

        self.trans_conv3 = nn.ConvTranspose2d(16, 2, kernel_size=3, stride=(2, 1), padding=1, output_padding=(1, 0))
        self.bn3 = nn.BatchNorm2d(2)

    def forward(self, x, snr):
        x = self.af1(self.prelu1(self.bn1(self.trans_conv1(x))), snr)
        x = self.af2(self.prelu2(self.bn2(self.trans_conv2(x))), snr)
        x = self.bn3(self.trans_conv3(x))
        return x

## Build the modules

In [ ]:
dataset_cfg = load_dataset_cfg(train_file)
atn = ATN().to(device)
encoder = CsiNetPlusEncoderWithAF(compression_ratio=cfg.compression_ratio).to(device)
channel_sim = WirelessChannelSimulator(num_bs_antennas=int(dataset_cfg['num_bs_antennas'])).to(device)
c2r = ComplexToReal().to(device)
decoder = CsiNetPlusDecoder(input_dim=encoder.M).to(device)
stn = STN().to(device)

all_params = (list(atn.parameters()) + list(encoder.parameters())
              + list(decoder.parameters()) + list(stn.parameters()))
print('Total trainable parameters:', sum(p.numel() for p in all_params if p.requires_grad))


## Training loop

In [ ]:
def samplewise_linear_nmse(h_true, h_pred):
    numerator = torch.sum((h_true - h_pred) ** 2, dim=(1, 2, 3))
    denominator = torch.sum(h_true ** 2, dim=(1, 2, 3)) + 1e-8
    return numerator / denominator

def spectral_correlation_rho(h_true, h_pred):
    r"""Differentiable approximation of the evaluator-style correlation metric.

    The tensors are expected in [B, 2, 32, 32] format where channel 0 is real
    and channel 1 is imaginary.
    """
    if h_true.shape != h_pred.shape:
        raise ValueError(f"Shape mismatch: h_true={h_true.shape}, h_pred={h_pred.shape}")

    # Move the real/imaginary dimension to the end: [B, 32, 32, 2]
    h_true = h_true.permute(0, 2, 3, 1).contiguous()
    h_pred = h_pred.permute(0, 2, 3, 1).contiguous()

    n = h_pred.size(0)
    nt = h_pred.size(1)
    nc = h_pred.size(2)
    nc_expand = 257

    if nc_expand < nc:
        raise ValueError(f"nc_expand ({nc_expand}) must be >= nc ({nc})")

    zeros_true = h_true.new_zeros((n, nt, nc_expand - nc, 2))
    zeros_pred = h_pred.new_zeros((n, nt, nc_expand - nc, 2))

    # Keep compatibility with older torch versions while using the modern path
    if version.parse(torch.__version__) > version.parse("1.7.0"):
        h_true_c = torch.view_as_complex(torch.cat((h_true, zeros_true), dim=2))
        h_pred_c = torch.view_as_complex(torch.cat((h_pred, zeros_pred), dim=2))

        raw_true = torch.view_as_real(torch.fft.fft(h_true_c))[:, :, :125, :]
        raw_pred = torch.view_as_real(torch.fft.fft(h_pred_c))[:, :, :125, :]
    else:
        h_true_padded = torch.cat((h_true, zeros_true), dim=2)
        h_pred_padded = torch.cat((h_pred, zeros_pred), dim=2)
        raw_true = torch.fft(h_true_padded, signal_ndim=1)[:, :, :125, :]
        raw_pred = torch.fft(h_pred_padded, signal_ndim=1)[:, :, :125, :]

    norm_pred = raw_pred[..., 0] ** 2 + raw_pred[..., 1] ** 2
    norm_pred = torch.sqrt(norm_pred.sum(dim=1) + 1e-12)

    norm_true = raw_true[..., 0] ** 2 + raw_true[..., 1] ** 2
    norm_true = torch.sqrt(norm_true.sum(dim=1) + 1e-12)

    real_cross = raw_pred[..., 0] * raw_true[..., 0] + raw_pred[..., 1] * raw_true[..., 1]
    real_cross = real_cross.sum(dim=1)
    imag_cross = raw_pred[..., 0] * raw_true[..., 1] - raw_pred[..., 1] * raw_true[..., 0]
    imag_cross = imag_cross.sum(dim=1)

    norm_cross = torch.sqrt(real_cross ** 2 + imag_cross ** 2 + 1e-12)
    rho = norm_cross / (norm_pred * norm_true + 1e-12)
    return rho

def forward_pass(H_d, H_u, snr):
    T = atn(H_d, snr)
    c = encoder(T, snr)
    s = enc_to_complex_and_normalize(c)
    s_hat = channel_sim(s, snr, H_u)
    c_hat = c2r(s_hat)
    T_hat = decoder(c_hat, snr)
    H_hat = stn(T_hat, snr)
    return H_hat

def compute_loss(H_true, H_pred, epoch_index):
    mse_loss = mse_criterion(H_pred, H_true)
    nmse_linear = samplewise_linear_nmse(H_true, H_pred).mean()
    rho = spectral_correlation_rho(H_true, H_pred).mean()
    corr_loss = 1.0 - rho

    if epoch_index < cfg.warmup_epochs:
        total_loss = mse_loss
    else:
        total_loss = (
            cfg.mse_weight * mse_loss
            + cfg.nmse_weight * nmse_linear
            + cfg.corr_weight * corr_loss
        )

    return total_loss, mse_loss.detach(), nmse_linear.detach(), rho.detach(), corr_loss.detach()

def run_epoch(split, epoch_index=0, fixed_snr=None):
    is_train = split == "train"
    models = [atn, encoder, decoder, stn, channel_sim]
    for model in models:
        model.train(is_train)

    rng = np.random.default_rng(SEED + epoch_index)
    total_loss = 0.0
    total_mse = 0.0
    total_rho = 0.0
    total_corr_loss = 0.0
    total_samples = 0
    nmse_all = []

    iterator = dataset.iterate_split(
        split,
        batch_size=cfg.batch_size,
        shuffle=is_train,
        generator=rng if is_train else None,
        fixed_snr=fixed_snr,
    )

    for batch_idx, (H_d, H_u, snr) in enumerate(iterator):
        if is_train and debug_train_batches is not None and batch_idx >= debug_train_batches:
            break
        if (not is_train) and debug_eval_batches is not None and batch_idx >= debug_eval_batches:
            break

        H_d = H_d.to(device, non_blocking=True)
        H_u = H_u.to(device, non_blocking=True)
        snr = snr.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            H_hat = forward_pass(H_d, H_u, snr)
            total_batch_loss, mse_loss, nmse_linear, rho, corr_loss = compute_loss(H_d, H_hat, epoch_index)

            if is_train:
                total_batch_loss.backward()
                if cfg.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(all_params, cfg.grad_clip)
                optimizer.step()

        batch_size = H_d.size(0)
        total_loss += float(total_batch_loss.detach().item()) * batch_size
        total_mse += float(mse_loss.item()) * batch_size
        total_rho += float(rho.item()) * batch_size
        total_corr_loss += float(corr_loss.item()) * batch_size
        total_samples += batch_size

        # NMSE is computed after denormalization to keep it in physical scale.
        H_d_denorm = dataset.denormalize(H_d.detach(), "dl")
        H_hat_denorm = dataset.denormalize(H_hat.detach(), "dl")
        nmse_batch = samplewise_linear_nmse(H_d_denorm, H_hat_denorm)
        nmse_all.extend(nmse_batch.detach().cpu().tolist())

    mean_loss = total_loss / max(total_samples, 1)
    mean_mse = total_mse / max(total_samples, 1)
    mean_rho = total_rho / max(total_samples, 1)
    mean_corr_loss = total_corr_loss / max(total_samples, 1)
    linear_nmse = float(np.mean(nmse_all)) if len(nmse_all) > 0 else float("nan")
    nmse_db = nmse_db_from_linear_nmse(linear_nmse) if len(nmse_all) > 0 else float("nan")

    return {
        "loss": mean_loss,
        "mse": mean_mse,
        "rho": mean_rho,
        "corr_loss": mean_corr_loss,
        "nmse_db": nmse_db,
        "linear_nmse": linear_nmse,
        "samples": total_samples,
    }

def save_checkpoint(epoch, best_linear_nmse, tag):
    checkpoint = {
        "epoch": epoch,
        "cfg": cfg.__dict__,
        "stats": stats,
        "best_linear_nmse": best_linear_nmse,
        "atn_state_dict": atn.state_dict(),
        "encoder_state_dict": encoder.state_dict(),
        "decoder_state_dict": decoder.state_dict(),
        "stn_state_dict": stn.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
    }
    path = Path(cfg.checkpoint_dir) / tag
    torch.save(checkpoint, path)
    print(f"Saved checkpoint: {path}")

def find_latest_checkpoint():
    checkpoint_dir = Path(cfg.checkpoint_dir)
    candidates = list(checkpoint_dir.glob("epoch_*.pth"))
    candidates += [checkpoint_dir / "final_model.pth", checkpoint_dir / "best_model.pth"]
    candidates = [path for path in candidates if path.exists()]
    if not candidates:
        raise FileNotFoundError(f"No checkpoints found in {checkpoint_dir}")
    return max(candidates, key=lambda path: path.stat().st_mtime)

def load_checkpoint(path):
    checkpoint = torch.load(path, map_location=device)
    atn.load_state_dict(checkpoint["atn_state_dict"])
    encoder.load_state_dict(checkpoint["encoder_state_dict"])
    decoder.load_state_dict(checkpoint["decoder_state_dict"])
    stn.load_state_dict(checkpoint["stn_state_dict"])
    if "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if "scheduler_state_dict" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    print(f"Loaded checkpoint: {path}")
    return checkpoint

def evaluate_snr_sweep(snr_points, split="test"):
    results = []
    for snr_db in snr_points:
        metrics = run_epoch(split, fixed_snr=snr_db)
        results.append(metrics["nmse_db"])
        print(
            f"SNR {snr_db:>4} dB -> NMSE {metrics['nmse_db']:.3f} dB "
            f"| ρ {metrics['rho']:.4f}"
        )
    return results

### Optimiser, scheduler and loss weights

These cells reproduce the variant's exact training recipe — open the source `.py` for the line-by-line argparse / CLI logic.

In [ ]:
# optimizer = optim.Adam(all_params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
#                                                   factor=0.5, patience=cfg.patience,
#                                                   min_lr=cfg.min_lr)
# mse_criterion = nn.MSELoss()
# (See the .py for any variant-specific overrides — e.g. cosine LR
#  schedules, AdamW, or per-parameter-group weight decay.)


### Run training

```python
for epoch in range(cfg.epochs):
    train_metrics = run_epoch('train', epoch_index=epoch)
    val_metrics   = run_epoch('val',   epoch_index=epoch)
    # scheduler.step(val_metrics['linear_nmse'])
```

After training, sweep test NMSE over a fixed SNR grid:

```python
snr_points = [-10, -5, 0, 5, 10]
nmse_db = evaluate_snr_sweep(snr_points, split='test')
```